# Cálculo da Taxa por 100 Mil Habitantes 

$$\text{Taxa}_{regional} = \frac{\sum \text{Quantidade dos eventos dos municípios da regional}}{\sum \text{População dos municípios da regional}} \times 100.000$$



### 1. Imports


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd()))

from utilitarios_comuns import (
    registrar_log,
    normalizar_codigo_ibge,
    normalizar_regional,
    extrair_ano_da_coluna,
    selecionar_colunas_de_valor,
    filtrar_populacao_valida_por_municipio,
    POPULACAO_MINIMA_VALIDA,
    SUFIXO_COLUNA_QUANTIDADE,
    PASTA_POPULACAO,
    CAMINHO_POPULACAO,
    SUFIXO_COLUNA_TOTAL,
    MULTIPLICADOR_TAXA,
    ESPECIFICACOES_VARIAVEIS,
    ESPECIFICACOES_VARIAVEIS_CATEGORICAS,
)

### Configurações

In [3]:
COLUNA_MUNICIPIO = "MUNICÍPIO"
COLUNA_CODIGO_IBGE = "CÓDIGO IBGE"
COLUNA_REGIONAL = "REGIONAL DE SAÚDE"
COLUNA_ANO = "ANO"
COLUNA_TIPO = "TIPO"
COLUNA_QUANTIDADE = "QUANTIDADE"
COLUNA_POPULACAO = "POPULACAO"

CASAS_DECIMAIS_TAXA = 2

PASTA_SAIDA = PASTA_POPULACAO

LISTA_VARIAVEIS = []
for nome, especificacao in ESPECIFICACOES_VARIAVEIS.items():
    if nome == "Hospitais":
        continue  # Hospitais tem processamento à parte (agrupa por TIPO também)
    configuracao_da_variavel = {"NOME": nome, **especificacao}
    configuracao_da_variavel["NOME_ARQUIVO_SAIDA_REGIONAL"] = especificacao["NOME_ARQUIVO_SAIDA_TAXA"]
    LISTA_VARIAVEIS.append(configuracao_da_variavel)

LISTA_VARIAVEIS_CATEGORICAS = []

for nome, especificacao in ESPECIFICACOES_VARIAVEIS_CATEGORICAS.items():
    configuracao_da_variavel = {"NOME": nome, **especificacao}
    configuracao_da_variavel["NOME_ARQUIVO_SAIDA_REGIONAL"] = especificacao["NOME_ARQUIVO_SAIDA_TAXA"]
    LISTA_VARIAVEIS_CATEGORICAS.append(configuracao_da_variavel)

CAMINHO_HOSPITAIS = ESPECIFICACOES_VARIAVEIS["Hospitais"]["CAMINHO_ENTRADA"]
ANO_REFERENCIA_HOSPITAIS = ESPECIFICACOES_VARIAVEIS["Hospitais"]["ANO_INICIAL"]
NOME_ARQUIVO_SAIDA_HOSPITAIS_REGIONAL = ESPECIFICACOES_VARIAVEIS["Hospitais"]["NOME_ARQUIVO_SAIDA_TAXA"]

### Carregamento da População (uma única vez)

In [4]:
tabela_populacao = pd.read_csv(CAMINHO_POPULACAO)
tabela_populacao[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_populacao[COLUNA_CODIGO_IBGE])
tabela_populacao[COLUNA_REGIONAL] = normalizar_regional(tabela_populacao[COLUNA_REGIONAL])

registrar_log(f"População carregada: {tabela_populacao.shape[0]} municípios.")
tabela_populacao.head()

[LOG] População carregada: 853 municípios.


,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1970_0-9,1970_10-19,1970_20-29,1970_30-39,1970_40-49,1970_50-59,1970_60-69,...,2016_TOTAL,2017_TOTAL,2018_TOTAL,2019_TOTAL,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL,2024_TOTAL,2025_TOTAL
0,abadia dos dourados,3100104,uberlandia,1542.0,1255.0,791.0,537.0,274.0,207.0,143.0,...,3277,3233,3195,3164,3142,3123,3111,3105,3106,3115
1,abaete,3100203,sete lagoas,3054.0,2786.0,1423.0,1115.0,874.0,470.0,311.0,...,11149,11177,11215,11263,11322,11392,11474,11569,11677,11799
2,abre campo,3100302,manhumirim,3015.0,2534.0,1462.0,1049.0,678.0,543.0,311.0,...,6956,6933,6914,6901,6895,6895,6903,6917,6936,6963
3,acaiaca,3100401,ponte nova,514.0,387.0,200.0,208.0,135.0,128.0,70.0,...,1918,1930,1942,1955,1974,1992,2012,2035,2057,2084
4,aguanil,3100807,divinopolis,359.0,302.0,217.0,187.0,126.0,75.0,23.0,...,2042,2065,2087,2112,2139,2167,2196,2228,2261,2297


## Calcular a taxa — agregando por Regional de Saúde antes de dividir

Dois cruzamentos, em pontos diferentes do processamento:

- **Por Código IBGE**, entre a base bruta de cada variável e a
  população: valida o par (município, ano) e é o que permite
  detectar e descartar, por ano, um município sem população válida
  sem descartar os demais anos do mesmo município (ver
  `pares_descartados` abaixo).
- **Por nome da Regional de Saúde** (já normalizado por
  `normalizar_regional`), depois de já ter somado quantidade e
  população por município: agrega ambas as somas ao nível de
  Regional antes de dividir, para que o resultado seja a razão das
  somas regionais, não uma soma de taxas municipais já calculadas.


In [ ]:
def calcular_taxa(quantidade, populacao):
    populacao_valida = populacao.where(populacao >= POPULACAO_MINIMA_VALIDA)
    return (quantidade / populacao_valida) * MULTIPLICADOR_TAXA


def calcular_taxa_por_regional(tabela_bruta, tabela_populacao, ano_inicial, ano_final,
                                colunas_de_agrupamento_extra=(), nome_da_variavel=""):
    colunas_de_agrupamento = [COLUNA_REGIONAL, *colunas_de_agrupamento_extra]

    municipios_sem_regional = tabela_bruta[COLUNA_REGIONAL].isna().sum()
    if municipios_sem_regional > 0:
        registrar_log(
            f"ATENÇÃO: {municipios_sem_regional} registro(s) sem Regional de Saúde "
            f"-- excluído(s) da agregação por regional."
        )
    tabela_bruta = tabela_bruta.dropna(subset=[COLUNA_REGIONAL]).copy()
    
    # Conta quantas linhas não têm Regional preenchida,
    colunas_de_valor = selecionar_colunas_de_valor(
        tabela_bruta, ano_inicial, ano_final,
        colunas_a_ignorar=(COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE, *colunas_de_agrupamento),
    )

    tabela_bruta[colunas_de_valor] = tabela_bruta[colunas_de_valor].astype(float)

    # Soma da população: todos os municípios da mesma regional somados,
    # por ano -- independente de quais colunas de valor a variável tem.

    colunas_de_populacao_no_periodo: list[str] = []
    for c in tabela_populacao.columns:
        if c.endswith(SUFIXO_COLUNA_TOTAL):
            colunas_de_populacao_no_periodo.append(c)
            
    tabela_populacao_filtrada = filtrar_populacao_valida_por_municipio(
        tabela_populacao, colunas_de_populacao_no_periodo
    )
    
    if tabela_populacao_filtrada[COLUNA_CODIGO_IBGE].duplicated().any():
        raise ValueError(
            "tabela_populacao contém CÓDIGO IBGE duplicado -- o alinhamento por "
            "código IBGE (.map) exige códigos únicos."
        )
    tabela_populacao_filtrada_indexada = tabela_populacao_filtrada.set_index(COLUNA_CODIGO_IBGE)

    colunas_por_ano = {}
    for coluna in colunas_de_valor:
        colunas_por_ano.setdefault(extrair_ano_da_coluna(coluna), []).append(coluna)

    pares_descartados = 0
    eventos_descartados = 0.0
    
    anos_afetados = []
    
    for ano, colunas_do_ano in colunas_por_ano.items():
        coluna_populacao_do_ano = f"{ano}{SUFIXO_COLUNA_TOTAL}"
        if coluna_populacao_do_ano not in tabela_populacao_filtrada_indexada.columns:
            continue  
        populacao_do_municipio_no_ano = tabela_bruta[COLUNA_CODIGO_IBGE].map(
            tabela_populacao_filtrada_indexada[coluna_populacao_do_ano]
        )
        mascara_populacao_invalida = populacao_do_municipio_no_ano.isna()
        if not mascara_populacao_invalida.any():
            continue
        mascara_com_quantidade = mascara_populacao_invalida & tabela_bruta[colunas_do_ano].notna().any(axis=1)
        if mascara_com_quantidade.any():

            colunas_para_contagem = (
                [coluna_populacao_do_ano] if coluna_populacao_do_ano in colunas_do_ano else colunas_do_ano
            )
            pares_descartados += int(mascara_com_quantidade.sum())
            eventos_descartados += float(
                tabela_bruta.loc[mascara_com_quantidade, colunas_para_contagem].sum().sum()
            )
            anos_afetados.append(ano)
        tabela_bruta.loc[mascara_populacao_invalida, colunas_do_ano] = np.nan

    if pares_descartados > 0 and nome_da_variavel:
        rotulo_par = "(" + ", ".join(["município", *colunas_de_agrupamento_extra, "ano"]) + ")"
        registrar_log(
            f"{nome_da_variavel}: {pares_descartados} par(es) {rotulo_par} com quantidade "
            f"descartada por população inválida antes da soma por regional -- "
            f"{eventos_descartados:g} evento(s) no total; ano(s) afetado(s): {sorted(anos_afetados)}."
        )

    quantidade_por_regional = (
        tabela_bruta.groupby(colunas_de_agrupamento, as_index=False)[colunas_de_valor]
        .sum(min_count=1)  # min_count=1: se TODOS os municípios tiverem NaN, o resultado continua NaN, não 0
    )

    populacao_por_regional = (
        tabela_populacao_filtrada.groupby(COLUNA_REGIONAL)[colunas_de_populacao_no_periodo]
        .sum(min_count=1)
    )

    colunas_de_taxa_calculadas = {}
    for coluna in colunas_de_valor:
        ano = extrair_ano_da_coluna(coluna)
        coluna_populacao_do_ano = f"{ano}{SUFIXO_COLUNA_TOTAL}"

        if coluna_populacao_do_ano not in populacao_por_regional.columns:
            registrar_log(
                f"ATENÇÃO: sem população para o ano {ano} -- coluna '{coluna}' ficará vazia (NaN)."
            )
            colunas_de_taxa_calculadas[coluna] = np.nan
            quantidade_por_regional[coluna] = np.nan
            continue

        populacao_da_regional_no_ano = quantidade_por_regional[COLUNA_REGIONAL].map(
            populacao_por_regional[coluna_populacao_do_ano]
        )
        taxa_da_coluna = calcular_taxa(quantidade_por_regional[coluna], populacao_da_regional_no_ano)
        colunas_de_taxa_calculadas[coluna] = taxa_da_coluna

    tabela_de_taxas = pd.DataFrame(colunas_de_taxa_calculadas).round(CASAS_DECIMAIS_TAXA)

    blocos = [quantidade_por_regional[colunas_de_agrupamento]]
    
    for coluna in colunas_de_valor:
        blocos.append(
            quantidade_por_regional[[coluna]].rename(columns={coluna: f"{coluna}{SUFIXO_COLUNA_QUANTIDADE}"})
        )
        blocos.append(tabela_de_taxas[[coluna]])
    return pd.concat(blocos, axis=1)


### Variáveis — Casos, Mortalidade, Mamografia, Escolaridade, Raça/Cor, Faixa Etária

Mesmo processamento pras variáveis simples e categóricas: só muda a
lista de configuração percorrida (a diferença entre elas está nas
colunas de valor de cada CSV de entrada, não no código).

In [6]:
for configuracao_da_variavel in LISTA_VARIAVEIS + LISTA_VARIAVEIS_CATEGORICAS:
    nome_da_variavel = configuracao_da_variavel["NOME"]
    registrar_log(f"===== INÍCIO: {nome_da_variavel.upper()} =====")

    tabela_bruta = pd.read_csv(configuracao_da_variavel["CAMINHO_ENTRADA"])
    tabela_bruta[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_bruta[COLUNA_CODIGO_IBGE])
    tabela_bruta[COLUNA_REGIONAL] = normalizar_regional(tabela_bruta[COLUNA_REGIONAL])

    tabela_com_taxa_regional = calcular_taxa_por_regional(
        tabela_bruta, tabela_populacao,
        configuracao_da_variavel["ANO_INICIAL"], configuracao_da_variavel["ANO_FINAL"],
        nome_da_variavel=nome_da_variavel,
    )

    caminho_de_saida_regional = PASTA_SAIDA / configuracao_da_variavel["NOME_ARQUIVO_SAIDA_REGIONAL"]
    caminho_de_saida_regional.parent.mkdir(parents=True, exist_ok=True)
    tabela_com_taxa_regional.to_csv(caminho_de_saida_regional, index=False)

    registrar_log(
        f"===== FIM: {nome_da_variavel.upper()} — salvo em {caminho_de_saida_regional.name} =====\n"
    )

[LOG] ===== INÍCIO: CASOS =====
[LOG] ATENÇÃO: 131 município(s) sem população válida (< 1) em ao menos um ano do período -- excluído(s) da soma por regional nesses anos.
[LOG] Casos: 698 par(es) (município, ano) com quantidade descartada por população inválida antes da soma por regional -- 0 evento(s) no total; ano(s) afetado(s): [1985, 1986, 1987, 1988, 1989, 1990].
[LOG] ===== FIM: CASOS — salvo em casos_ano_diagnostico_taxa.csv =====

[LOG] ===== INÍCIO: MORTALIDADE =====
[LOG] ATENÇÃO: 131 município(s) sem população válida (< 1) em ao menos um ano do período -- excluído(s) da soma por regional nesses anos.


/tmp/ipykernel_17908/3120363214.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .sum(min_count=1)  # min_count=1: se TODOS os municípios tiverem NaN, o resultado continua NaN, não 0


[LOG] ===== FIM: MORTALIDADE — salvo em mortalidade_taxa.csv =====

[LOG] ===== INÍCIO: MAMOGRAFIA =====
[LOG] ATENÇÃO: 6 registro(s) sem Regional de Saúde -- excluído(s) da agregação por regional.
[LOG] ATENÇÃO: 131 município(s) sem população válida (< 1) em ao menos um ano do período -- excluído(s) da soma por regional nesses anos.


/tmp/ipykernel_17908/3120363214.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .sum(min_count=1)  # min_count=1: se TODOS os municípios tiverem NaN, o resultado continua NaN, não 0


[LOG] ===== FIM: MAMOGRAFIA — salvo em mamografia_taxa.csv =====

[LOG] ===== INÍCIO: CASOS POR ESCOLARIDADE =====
[LOG] ATENÇÃO: 131 município(s) sem população válida (< 1) em ao menos um ano do período -- excluído(s) da soma por regional nesses anos.


/tmp/ipykernel_17908/3120363214.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .sum(min_count=1)  # min_count=1: se TODOS os municípios tiverem NaN, o resultado continua NaN, não 0


[LOG] ===== FIM: CASOS POR ESCOLARIDADE — salvo em casos_escolaridade_taxa.csv =====

[LOG] ===== INÍCIO: CASOS POR RAÇA E COR =====
[LOG] ATENÇÃO: 131 município(s) sem população válida (< 1) em ao menos um ano do período -- excluído(s) da soma por regional nesses anos.


/tmp/ipykernel_17908/3120363214.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .sum(min_count=1)  # min_count=1: se TODOS os municípios tiverem NaN, o resultado continua NaN, não 0


[LOG] ===== FIM: CASOS POR RAÇA E COR — salvo em casos_raca_cor_taxa.csv =====

[LOG] ===== INÍCIO: CASOS POR FAIXA ETÁRIA =====
[LOG] ATENÇÃO: 131 município(s) sem população válida (< 1) em ao menos um ano do período -- excluído(s) da soma por regional nesses anos.


/tmp/ipykernel_17908/3120363214.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .sum(min_count=1)  # min_count=1: se TODOS os municípios tiverem NaN, o resultado continua NaN, não 0


[LOG] ===== FIM: CASOS POR FAIXA ETÁRIA — salvo em casos_faixa_etaria_taxa.csv =====



### Variável Hospital

Agrupa por (`REGIONAL DE SAÚDE`, `TIPO`) em vez de só `REGIONAL DE
SAÚDE` — cada tipo de habilitação continua em uma linha própria, sem
somar Oncologia com Reconstrução Mamária.

O mesmo CSV de saída também recebe uma linha por regional com
`TIPO = "TODOS"` (soma de todos os tipos de habilitação). Os
gráficos filtram por `TIPO` antes de somar; qualquer outro consumo
deste CSV precisa filtrar `TIPO` também, ou vai contar cada hospital
em dobro (uma vez por tipo específico, outra em "TODOS").

In [7]:
tabela_hospitais_bruta = pd.read_csv(CAMINHO_HOSPITAIS)
tabela_hospitais_bruta[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_hospitais_bruta[COLUNA_CODIGO_IBGE])
tabela_hospitais_bruta[COLUNA_REGIONAL] = normalizar_regional(tabela_hospitais_bruta[COLUNA_REGIONAL])

tabela_hospitais_por_tipo = calcular_taxa_por_regional(
    tabela_hospitais_bruta, tabela_populacao,
    ANO_REFERENCIA_HOSPITAIS, ANO_REFERENCIA_HOSPITAIS,
    colunas_de_agrupamento_extra=(COLUNA_TIPO,),
    nome_da_variavel="Hospitais",
)

# Mesma taxa, mas somando todos os tipos de habilitação por regional.
tabela_hospitais_total = calcular_taxa_por_regional(
    tabela_hospitais_bruta, tabela_populacao,
    ANO_REFERENCIA_HOSPITAIS, ANO_REFERENCIA_HOSPITAIS,
    nome_da_variavel="Hospitais (Total)",
)
tabela_hospitais_total.insert(1, COLUNA_TIPO, "TODOS")

tabela_hospitais_com_taxa_regional = pd.concat(
    [tabela_hospitais_por_tipo, tabela_hospitais_total], ignore_index=True
).sort_values([COLUNA_REGIONAL, COLUNA_TIPO]).reset_index(drop=True)

caminho_de_saida_hospitais_regional = PASTA_SAIDA / NOME_ARQUIVO_SAIDA_HOSPITAIS_REGIONAL
caminho_de_saida_hospitais_regional.parent.mkdir(parents=True, exist_ok=True)
tabela_hospitais_com_taxa_regional.to_csv(caminho_de_saida_hospitais_regional, index=False)

registrar_log(f"Hospitais salvo em {caminho_de_saida_hospitais_regional.name}")
tabela_hospitais_com_taxa_regional.head()

[LOG] ATENÇÃO: 131 município(s) sem população válida (< 1) em ao menos um ano do período -- excluído(s) da soma por regional nesses anos.
[LOG] ATENÇÃO: 131 município(s) sem população válida (< 1) em ao menos um ano do período -- excluído(s) da soma por regional nesses anos.
[LOG] Hospitais salvo em hospitais_taxa.csv


,REGIONAL DE SAÚDE,TIPO,2023_TOTAL_QUANTIDADE,2023_TOTAL
0,alfenas,ONCOLOGIA,1.0,0.42
1,alfenas,RECONSTRUÇÃO MAMÁRIA,1.0,0.42
2,alfenas,TODOS,2.0,0.84
3,barbacena,ONCOLOGIA,1.0,0.38
4,barbacena,TODOS,1.0,0.38
